In [ ]:
from pathlib import Path
import sys

project_root = Path().resolve().parent
sys.path.append(str(project_root)) 

In [ ]:
from src.schema import ConfigSchema
from src.utils.config import read_yaml

raw_config = read_yaml(project_root / "configs" / "config.yaml")
media_root = project_root / "media"
log_root = project_root / "logs"
config = ConfigSchema(**raw_config)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# Настройки для векторного экспорта (текст не переводится в кривые)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

mpl.rcParams['font.family'] = 'Times New Roman'
mpl.rcParams['font.size'] = 9  # Среднее значение (можно 8 или 10)

# Parse

In [ ]:
import re
import pandas as pd

LOSS_RE = re.compile(
    r"(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})"
    r".*?Round (?P<round>\d+)\s*\|\s*"
    r"loss avg=(?P<avg>[\d\.]+)\s+"
    r"min=(?P<min>[\d\.]+)\s+"
    r"max=(?P<max>[\d\.]+)\s+"
    r"batches=(?P<batches>\d+)"
)

def parse_logs(log_lines):
    rows = []

    for line in log_lines:
        m = LOSS_RE.search(line)
        if m:
            rows.append({
                "ts": pd.to_datetime(m["ts"]),
                "round": int(m["round"]),
                "loss_avg": float(m["avg"]),
                "loss_min": float(m["min"]),
                "loss_max": float(m["max"]),
                "batches": int(m["batches"]),
            })

    return pd.DataFrame(rows)

In [ ]:
with open(log_root / "splitfed_asr.log") as f:
    l = f.readlines()
    df = parse_logs(l)

In [ ]:
last_run = df.groupby(by="round").aggregate("last").reset_index()

In [ ]:
# df = last_run.loc[3:89]

plt.figure(figsize=(10, 5))

plt.plot(last_run["round"], last_run["loss_avg"], marker="o", linewidth=2)

plt.title("Динамика функции потерь в процессе обучения")
plt.xlabel("Раунд")
plt.ylabel("Ср. ошибка")
plt.ylim(0, 1.5)

plt.grid(True, alpha=0.3)

#Экспорт в векторные форматы
plt.savefig(f'{media_root}/sfl_loss_3_plots.pdf', format='pdf', bbox_inches='tight', dpi=300)
plt.savefig(f'{media_root}/sfl_loss_3_plots.svg', format='svg', bbox_inches='tight')
plt.savefig(f'{media_root}/sfl_loss_3_plots.eps', format='eps', bbox_inches='tight')

plt.show()

# Eval

In [ ]:
from src.dataset.analytics import DataConcatenator

In [ ]:
df_0 = pd.read_csv(project_root / "experiments/results/Client0_eval.csv")
df_1 = pd.read_csv(project_root / "experiments/results/Client1_eval.csv")
df_2 = pd.read_csv(project_root / "experiments/results/Client2_eval.csv")

In [ ]:
df_c = pd.concat([df_0, df_1, df_2])

from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

f1 = f1_score(df_c.labels, df_c.preds)
acc = accuracy_score(df_c.labels, df_c.preds)
prec = precision_score(df_c.labels, df_c.preds)
reca = recall_score(df_c.labels, df_c.preds)

In [ ]:
print(acc, f1, prec, reca)

In [ ]:
datasets = [
    ("CREMA-D (Client0)", df_0),
    ("RAVDESS (Client1)", df_1),
    ("SAVEE (Client2)", df_2),
]

for name, df in datasets:
    plt.figure()
    plt.hist(df["probs"].values, bins=50)
    plt.title(f"Предсказанное распределение вероятностей — {name}")
    plt.xlabel("Вероятность")
    plt.ylabel("Частота")
    plt.grid(True)
    plt.show()

# Preprocess

In [ ]:
dataset_configs = [client.dataset for client in config.clients]

data = DataConcatenator(dataset_configs)

In [ ]:
all_data, all_meta = data.get_agg_data()

In [ ]:
df = data.to_dataframe(all_data, all_meta)

In [ ]:
df.to_csv(project_root / "data" / "preprocess" / "aggregated_data.csv", index=False)

In [ ]:
df = pd.read_csv(project_root / "data" / "preprocess" / "aggregated_data.csv")

In [ ]:
feature_cols = [
    c for c in df.columns
    if c not in ["label", "actor_id", "sex", "dataset"]
]

stats = df.groupby("dataset")[feature_cols].agg(["mean", "std"])

stats.columns = ["_".join(col) for col in stats.columns]

stats = stats.reset_index()

In [ ]:
df = df.merge(stats, on="dataset", how="left")

In [ ]:
for col in feature_cols:
    df[col] = (
        df[col] - df[f"{col}_mean"]
    ) / (df[f"{col}_std"] + 1e-8)

In [ ]:
df = df.drop(columns=[
    f"{c}_mean" for c in feature_cols
] + [
    f"{c}_std" for c in feature_cols
])

In [ ]:
sns.kdeplot(
    data=df,
    x="mfcc_1_mean",
    hue="label",
    common_norm=False,
    fill=True,
    alpha=0.3
)

In [ ]:
fig, axes = plt.subplots(
    1, 3,
    figsize=(14, 4),
    sharey=True,
    sharex=True
)

datasets = [
    "DatasetType.crema_d",
    "DatasetType.ravdess",
    "DatasetType.savee"
]

titles = ["CREMA-D", "RAVDESS", "SAVEE"]

for ax, ds, title in zip(axes, datasets, titles):
    temp = df[df["dataset"] == ds]
    sns.kdeplot(
        data=temp,
        x="mfcc_1_mean",
        hue="label",
        fill=True,
        alpha=0.30,
        common_norm=False,
        ax=ax,
        legend=False
    )
    ax.set_title(title)
    ax.set_xlabel("Значение MFCC 1")
    ax.set_ylabel("Плотность")

fig.legend(
    ['Конфликтная', 'Неконфликтная'],
    title="Метка класса",
    loc="center right",
    bbox_to_anchor=(1.1, 0.5),
    fontsize=10,
    title_fontsize=11,
    frameon=False
)

plt.tight_layout()

#Экспорт в векторные форматы
plt.savefig(f'{media_root}/kde_plots.pdf', format='pdf', bbox_inches='tight', dpi=300)
plt.savefig(f'{media_root}/kde_plots.svg', format='svg', bbox_inches='tight')
plt.savefig(f'{media_root}/kde_plots.eps', format='eps', bbox_inches='tight')

plt.show()

In [ ]:
ax = sns.boxplot(
    data=df,
    x="label",
    y="mfcc_1_mean",
    hue="dataset"
)
ax.set_xlabel("Метка класса")
ax.set_ylabel("Значение MFCC1")

handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, ["CREMA-D", "RAVDESS", "SAVEE"], title="Набор данных")